<a href="https://colab.research.google.com/github/mvishrutha/AI-LAB-1BF24CS157-M-Vishrutha-/blob/main/Week5_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def is_variable(x):
    return isinstance(x, str) and x.startswith("?")

def occurs(var, expr, subst):
    expr = apply_substitution(expr, subst)

    if var == expr:
        return True

    if isinstance(expr, tuple):
        return any(occurs(var, arg, subst) for arg in expr[1:])

    return False


def apply_substitution(expr, subst):
    if is_variable(expr):
        if expr in subst:
            return apply_substitution(subst[expr], subst)
        return expr

    if isinstance(expr, tuple):
        return (expr[0],) + tuple(
            apply_substitution(arg, subst) for arg in expr[1:]
        )

    return expr


def unify(x1, x2, subst=None):
    if subst is None:
        subst = {}

    x1 = apply_substitution(x1, subst)
    x2 = apply_substitution(x2, subst)

    # Step 1: If both expressions are identical
    if x1 == x2:
        return subst

    # Step 2: If x1 is a variable
    if is_variable(x1):
        if occurs(x1, x2, subst):
            return None
        # Create a copy to prevent in-place mutation side effects across branches if needed,
        # but for standard sequential unification we can update and return
        subst[x1] = x2
        return subst

    # Step 3: If x2 is a variable
    if is_variable(x2):
        if occurs(x2, x1, subst):
            return None
        subst[x2] = x1
        return subst

    # Step 4: Check whether both are compound expressions
    if isinstance(x1, tuple) and isinstance(x2, tuple):
        # Check predicate/function symbols
        if x1[0] != x2[0]:
            return None

        # Check number of arguments
        if len(x1) != len(x2):
            return None

        # Step 5: Unify corresponding arguments
        for a, b in zip(x1[1:], x2[1:]):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst

    # Different constants
    return None


# Helper function to print test results clearly
def run_test(case_name, expr1, expr2):
    print(f"--- {case_name} ---")
    print(f"Expr 1: {expr1}")
    print(f"Expr 2: {expr2}")
    result = unify(expr1, expr2)
    if result is None:
        print("Result: FAILURE (Not Unifiable)")
    else:
        print("MGU:", {k: apply_substitution(v, result) for k, v in result.items()})
    print()


# User requested test cases
# 1. P(f(a), g(y)) and P(X, X)
# Note: x is capitalized in prompt, treated as variable ?x
run_test(
    "Case 1: {P(f(a), g(y)), P(X, X)}",
    ("P", ("f", "a"), ("g", "?y")),
    ("P", "?x", "?x")
)

# 2. Q(a, g(x, a), f(y)) and Q(a, g(f(b), a), x)
# variables: ?x, ?y
run_test(
    "Case 2: {Q(a, g(x, a), f(y)), Q(a, g(f(b), a), x)}",
    ("Q", "a", ("g", "?x", "a"), ("f", "?y")),
    ("Q", "a", ("g", ("f", "b"), "a"), "?x")
)

# 3. P(b, x, f(g(z))) and P(z, f(y), f(y))
# variables: ?x, ?y, ?z
run_test(
    "Case 3: {P(b, x, f(g(z))), P(z, f(y), f(y))}",
    ("P", "b", "?x", ("f", ("g", "?z"))),
    ("P", "?z", ("f", "?y"), ("f", "?y"))
)

--- Case 1: {P(f(a), g(y)), P(X, X)} ---
Expr 1: ('P', ('f', 'a'), ('g', '?y'))
Expr 2: ('P', '?x', '?x')
Result: FAILURE (Not Unifiable)

--- Case 2: {Q(a, g(x, a), f(y)), Q(a, g(f(b), a), x)} ---
Expr 1: ('Q', 'a', ('g', '?x', 'a'), ('f', '?y'))
Expr 2: ('Q', 'a', ('g', ('f', 'b'), 'a'), '?x')
MGU: {'?x': ('f', 'b'), '?y': 'b'}

--- Case 3: {P(b, x, f(g(z))), P(z, f(y), f(y))} ---
Expr 1: ('P', 'b', '?x', ('f', ('g', '?z')))
Expr 2: ('P', '?z', ('f', '?y'), ('f', '?y'))
MGU: {'?z': 'b', '?x': ('f', ('g', 'b')), '?y': ('g', 'b')}

